A script implementing BICePs to reweight populations for a simple three state
toy model system.  Here, our prior comes from random generation of the Boltzmann
distribution and reweighting is performed using two experimental observables both set to 0.0 A.U.

For more details about this toy model systema and visual aids, please refer to
this notebook: `examples/enforcing_uniform_reference.ipynb`

In [1]:
import sys, os
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
import pandas as pd
from sklearn import metrics
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import biceps
from biceps.PosteriorSampler import u_kln_and_states_kn
from pymbar import MBAR

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [2]:
class Data:
    def __init__(self, array_list):
        self.array_list = array_list

    def save(self, filename):
        with open(filename, 'wb') as f:
            pickle.dump(self.array_list, f)

    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            array_list = pickle.load(f)
        return cls(array_list)

In [3]:
def write_noe_files(weights, x, exp, dir):
    for i in range(len(weights)):
        model = pd.read_pickle("template.noe")
        _model = pd.DataFrame()

        for j in range(len(exp)):
            row = model.iloc[0].copy()
            row["restraint_index"] = int(exp[j][0])
            row["atom_index1"] = int(exp[j][1])
            row["atom_index2"] = int(exp[j][2])
            row["exp"] = float(exp[j][3])
            row["model"] = float(x[i][j])  # x[i] must be flat and match len(exp)
            _model = pd.concat([_model, row.to_frame().T], ignore_index=True)

        _model.to_pickle(f"{dir}/{i}.noe")


###### Parameters #######

In [4]:
nStates,Nd = 156,1 # 156 distance with 1 NOE observables
n_xis,n_lambdas,nreplicas,nsteps,change_Nr_every,swap_every=1,2,1,1000000,0,0
multiprocess=4
σ_prior=0.161 # 0.08, 0.16
stat_model,data_uncertainty="Students","single"
data_likelihood = "gaussian" #"log normal" # "gaussian"

write_every = 10
attempt_move_state_every = 1
attempt_move_sigma_every = 1

Make output directories

In [5]:
state_dir = f"{nStates}_state"
biceps.toolbox.mkdir(state_dir)

datapoints_dir = f"{state_dir}/{nStates}_state_{Nd}_datapoints"
biceps.toolbox.mkdir(datapoints_dir)

dir = f"{datapoints_dir}/Prior_error_{σ_prior}"
biceps.toolbox.mkdir(dir)

## Loaded the population

In [6]:
clustering = pd.read_csv(f"../clustering/cluster_percentages.csv")

populations = np.array(clustering["Population"])
populations.shape

(296,)

In [7]:
energies = -np.log(populations)
energies.shape

(296,)

## Load the Prior Model (From MD) Calculated NOE distances 

In [8]:
md_distances = pd.read_csv(f"../clustering/md_distances.csv")

forward_model_data = np.array(md_distances)
forward_model_data.shape

(296, 156)

## Load the Refer NMR (Experimental Measurement)

In [9]:
restraints_table = {
    'weak': 5,
    'medium': 3.5,
    'strong': 2.5
}

dist_res_file = '../../../../utils/nspe_10_restraints.csv'
df_dist_res = pd.read_csv(dist_res_file)

for i, row in df_dist_res.iterrows():
    df_dist_res.at[i, df_dist_res.columns[2]] = restraints_table[row[2]]  # Correct mapping

# Make a look up table for intensity 
df_dist_res
dist_res = df_dist_res.values.tolist()
dist_res[:5]

[[24, 13, 5], [24, 14, 5], [25, 13, 5], [25, 14, 5], [137, 13, 5]]

In [10]:
np.shape(dist_res)

(156, 3)

In [11]:
### Extract the restraints_index from the labels 

human_readable_labels = "../clustering/human_readable_labels.csv"
df_labels = pd.read_csv(human_readable_labels)
df_labels[:5]
restraint_index = pd.factorize(df_labels['0'])[0] + 1

In [12]:
## Create experiment input with a list of [restraint_index, atom_index1, atom_index2, exp]

exp = [[r] + d for r, d in zip(restraint_index, dist_res)]
exp[:5]

type(exp)

list

## Write the NOE files 

In [13]:
data_dir = f"{dir}/NOE"
biceps.toolbox.mkdir(data_dir)

write_noe_files(weights=energies, x=forward_model_data, exp=exp, dir=data_dir)

In [14]:
df = pd.read_pickle(f"{data_dir}/1.noe")

df[:7]

,restraint_index,atom_index1,res1,atom_name1,atom_index2,res2,atom_name2,exp,model,comments
0,1,24,UNK1,H1,13,UNK1,H20,5.0,2.602989,NaN
1,1,24,UNK1,H1,14,UNK1,H20,5.0,2.262157,NaN
2,1,25,UNK1,H1,13,UNK1,H20,5.0,2.952681,NaN
3,1,25,UNK1,H1,14,UNK1,H20,5.0,2.581653,NaN
4,2,137,UNK1,H1,13,UNK1,H20,5.0,6.965965,NaN
5,2,137,UNK1,H1,14,UNK1,H20,5.0,5.909644,NaN
6,2,138,UNK1,H1,13,UNK1,H20,5.0,7.893921,NaN


## Load the input data

In [15]:
input_data = biceps.toolbox.sort_data(data_dir)
print(f"Input data: {biceps.toolbox.list_extensions(input_data)}")
forward_model_data = np.array([pd.read_pickle(i)["model"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/*.noe")])
experiment = np.array([pd.read_pickle(i)["exp"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/0.noe")])[0]


Input data: ['.noe']


In [16]:
outdir = f'{dir}/{stat_model}_{data_uncertainty}_sigma/{nsteps}_steps_{nreplicas}_replicas_{n_lambdas}_lam__swap_every_{swap_every}'
biceps.toolbox.mkdir(outdir)
print(f"nSteps of sampling: {nsteps}\nnReplicas: {nreplicas}")
lambda_values = np.linspace(0.0, 1.0, n_lambdas)

nSteps of sampling: 1000000
nReplicas: 1


In [17]:
sigMin,sigMax,dsig = 0.001,200,1.02
arr = np.exp(np.arange(np.log(sigMin), np.log(sigMax), np.log(dsig)))
l = len(arr)
sigma_index = round(l*0.73)

In [18]:
beta,beta_index=(1., 2.0, 1),0
_arr = np.linspace(*beta)
_l = len(_arr)
print("Alpha starts here: ",_arr[beta_index])
phi,phi_index=(1., 2.0, 1),0
gamma,gamma_index=(1.0, 2.0, np.e),0

Alpha starts here:  1.0


In [19]:
options = [dict(ref="uniform", stat_model=stat_model,
            sigma=(sigMin, sigMax, dsig), sigma_index=sigma_index, gamma=gamma,
            beta=beta, beta_index=beta_index, phi=phi, phi_index=phi_index,
            data_uncertainty=data_uncertainty, data_likelihood=data_likelihood,
            )]
print(pd.DataFrame(options))


       ref stat_model               sigma  sigma_index  \
0  uniform   Students  (0.001, 200, 1.02)          450   

                           gamma           beta  beta_index            phi  \
0  (1.0, 2.0, 2.718281828459045)  (1.0, 2.0, 1)           0  (1.0, 2.0, 1)   

   phi_index data_uncertainty data_likelihood  
0          0           single        gaussian  


In [20]:
ensemble = biceps.ExpandedEnsemble(lambda_values=lambda_values, energies=energies)
ensemble.initialize_restraints(input_data, options, verbose=1)
print("ensemble.expanded_values = ",ensemble.expanded_values)

Time to initalize restraints: 0.66s
ensemble.expanded_values =  [(0.0, 1.0), (1.0, 1.0)]


In [21]:
sampler = biceps.PosteriorSampler(ensemble, nreplicas, change_Nr_every, write_every=write_every)
sampler.sample(nsteps, attempt_lambda_swap_every=swap_every, swap_sigmas=1,
        attempt_move_state_every=attempt_move_state_every,
        attempt_move_sigma_every=attempt_move_sigma_every,
        verbose=0, progress=1, multiprocess=True, capture_stdout=0)

 ██████████████████████████████▏ 100.0% [1000000/1000000 | 34.5 kHz | 1 | 0s | 29s] MCMC 


In [22]:
expanded_values = sampler.expanded_values
A = biceps.Analysis(sampler, outdir=outdir, nstates=len(energies), MBAR=True, multiprocess=False, capture_stdout=0)
A.plot(plottype="step", figsize=(12,14), figname=f"BICePs.pdf", pad=0.35, plot_all_distributions=1)
plt.show()
A.plot_energy_trace()
plt.show()
BS, pops = A.f_df, A.P_dP[:,len(expanded_values[:])-1]
BS /= sampler.nreplicas
K = len(expanded_values[:])-1
pops_std = A.P_dP[:,2*K]
print(f"Predicted populatins: {pops}")

These states have not been sampled:
 [  3   7   9  10  12  14  16  17  18  20  21  22  24  25  26  27  28  30
  31  33  34  35  36  37  38  39  40  41  42  44  45  47  48  49  50  51
  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69
  70  71  72  73  74  75  77  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  94  95  96  97  98  99 100 101 103 104 105 106 107 108 109
 110 111 112 113 114 115 117 118 120 121 123 125 126 127 128 129 130 131
 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 148 149 151
 152 154 155 157 158 159 160 161 162 163 164 165 167 168 169 170 171 172
 173 174 175 176 177 178 180 181 183 185 186 187 188 189 190 191 192 193
 194 195 196 197 198 199 202 203 204 205 206 207 208 209 211 212 213 214
 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 233
 234 235 236 237 238 239 241 243 246 247 248 249 250 251 252 253 254 255
 256 260 261 264 266 267 268 269 271 272 273 275 278 279 281 282 283 284
 285 286 289 2


******* JAX 64-bit mode is now on! *******
*     JAX is now set to 64-bit mode!     *
*   This MAY cause problems with other   *
*      uses of JAX in the same code.     *
******************************************



 ██████████████████████████████▏ 100.0% [100000/100000 | 162.4 kHz | 1 | 0s | 1s] u_kln 
Time for MBAR: 10.878 s
Writing 156_state/156_state_1_datapoints/Prior_error_0.161/Students_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/BS.dat...
Writing 156_state/156_state_1_datapoints/Prior_error_0.161/Students_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/populations.dat...
Top 29 states: [182, 245, 43, 200, 0, 11, 287, 244, 201, 23, 258, 124, 263, 8, 280, 104, 232, 274, 179, 262, 184, 240, 242, 156, 166, 78, 2, 6, 153]
Top 29 populations: [2.69368424e-06 3.02483742e-06 4.84742548e-06 5.19750913e-06
 5.88278366e-06 1.00823654e-05 1.10275155e-05 2.32095793e-05
 2.40443214e-05 3.41080469e-05 3.63759960e-05 4.01019534e-05
 4.27517961e-05 5.05536232e-05 8.09808252e-05 1.02329795e-04
 1.14387520e-04 5.93461641e-04 9.09434555e-04 1.31843461e-03
 1.66931300e-03 2.57027415e-03 3.46507766e-03 1.28946442e-02
 2.10643230e-02 1.12982281e-01 1.65583506e-01 2.82722504e-01
 3.9360835

/var/folders/d8/y2dvs1ln1gjcwccrkvtffr240000gn/T/ipykernel_94387/3964779525.py:4: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Predicted populatins: [5.88278366e-06 1.00267122e-06 1.65583506e-01 0.00000000e+00
 8.52080300e-07 1.75966855e-06 2.82722504e-01 0.00000000e+00
 5.05536232e-05 0.00000000e+00 0.00000000e+00 1.00823654e-05
 0.00000000e+00 1.78909187e-06 0.00000000e+00 3.10138455e-07
 0.00000000e+00 0.00000000e+00 0.00000000e+00 1.32529517e-06
 0.00000000e+00 0.00000000e+00 0.00000000e+00 3.41080469e-05
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 2.21282367e-08 0.00000000e+00 0.00000000e+00
 3.68462758e-07 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 4.84742548e-06
 0.00000000e+00 0.00000000e+00 8.61350376e-07 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0

In [23]:
pops.shape

# Convert to DataFrame
df_predicted_population = pd.DataFrame(pops)

# Save to CSV
df_predicted_population.to_csv("../clustering/predicted_population_biceps.csv", index=False)

In [24]:
most_populated_index = np.argmax(pops)
most_populated_value = pops[most_populated_index]

print(f"Most populated state: {most_populated_index} with value {most_populated_value}")


Most populated state: 153 with value 0.3936083590615054


In [25]:
top5_indices = np.argsort(pops)[-5:][::-1]  # Sort, take last 5, reverse for descending order
top5_values = pops[top5_indices]

for i, (idx, val) in enumerate(zip(top5_indices, top5_values), 1):
    print(f"Top {i}: index = {idx}, population = {val:.4f}")

top5_indices

Top 1: index = 153, population = 0.3936
Top 2: index = 6, population = 0.2827
Top 3: index = 2, population = 0.1656
Top 4: index = 78, population = 0.1130
Top 5: index = 166, population = 0.0211


array([153,   6,   2,  78, 166])